In [26]:
import os
import json

import numpy as np
import pandas as pd

In [27]:
runs = [
    os.path.join('experiments', str(x)) for x in range(1981, 2021)
]

assert len(runs) == 40, len(runs)

In [28]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [29]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    if os.path.exists(os.path.join(run, 'init_time.txt')):
        init_time = float(open(os.path.join(run, 'init_time.txt')).read())
    else:
        init_time = 0.
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item() + init_time
    
    
    df.loc[len(df)] = info

In [30]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed',
    'noise_scale'
]].rename(columns={
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC',
    'noise_scale': 'Noise Scale'
}).groupby(by=['Solution', 'w/ BC', 'Noise Scale']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2f}'.format) + ' ± ' + sem.map('{:.2f}'.format)
        

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to min. RL2E. (s)  \
Solution    w/ BC Noise Scale                                                
Plane waves False 0.00001                25.2 ± 2.6             68.6 ± 0.3   
                  0.00010                25.0 ± 2.2             69.2 ± 0.5   
                  0.00100                25.6 ± 1.6             69.1 ± 0.3   
                  0.01000                83.2 ± 4.3             83.4 ± 4.2   
            True  0.00001                29.6 ± 1.4             68.1 ± 0.9   
                  0.00010                28.9 ± 0.8             68.2 ± 0.6   
                  0.00100                28.3 ± 0.7             69.0 ± 0.4   
                  0.01000                42.4 ± 2.8             68.5 ± 0.5   

                              Min. RL2E. (%)  
Solution    w/ BC Noise Scale                 
Plane waves False 0.00001        0.02 ± 0.00  
                  0.00010        0.02 ± 0.00  
                  0.00100        0.02 ± 0.00  
                  0.01000        0.05 ± 0.00  
            True  0.00001        0.02 ± 0.00  
                  0.00010        0.02 ± 0.00  
                  0.00100        0.02 ± 0.00  
                  0.01000        0.04 ± 0.00